In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import vectorbt as vbt
import sys

sys.path.insert(0, str(Path('.').resolve().parent))
from dema_strat import load_ohlcv, vbt_frame, free

CASH = 50_000
COMMISSION = 0.00015
TABLE_NAME = "f0_raw_smoothing"
SMOOTH_PERIOD = 12

def load_data(file_path, *, verbose=False, use_vwap=False):
    # Compact float32 + Parquet-cached loader.
    return load_ohlcv(file_path, use_cache=True, add_vwap=use_vwap, verbose=verbose)

def make_signals(data, n1=9, atr_period=14, sl_mult=1.0, rr_ratio=0.5, use_vwap=False, smooth_period=SMOOTH_PERIOD):
    # keep float32 through indicator math; cast up only at the vectorbt boundary
    close = data["Close"]
    high = data["High"]
    low = data["Low"]

    ema1 = vbt.indicators.MA.run(close, window=n1, ewm=True)
    smoothed = vbt.indicators.MA.run(ema1.ma, window=smooth_period, ewm=True)
    atr = vbt.indicators.ATR.run(high, low, close, window=atr_period)

    sl_dist = sl_mult * atr.atr
    tp_dist = rr_ratio * sl_dist
    sl_pct = sl_dist / close
    tp_pct = tp_dist / close

    long_mask = (smoothed.ma > ema1.ma) & np.isfinite(atr.atr) & (atr.atr > 0)
    short_mask = (ema1.ma > smoothed.ma) & np.isfinite(atr.atr) & (atr.atr > 0)

    if use_vwap:
        long_mask = long_mask & (close > data["VWAP"])
        short_mask = short_mask & (close < data["VWAP"])
    long_signal = long_mask
    short_signal = short_mask

    return pd.DataFrame({
        "ema1": ema1.ma,
        "smoothed": smoothed.ma,
        "atr": atr.atr,
        "sl_pct": sl_pct,
        "tp_pct": tp_pct,
        "long": long_signal,
        "short": short_signal,
    })

def run_backtest(data, n1=9, atr_period=14, sl_mult=1.0, rr_ratio=0.5, use_vwap=False, smooth_period=SMOOTH_PERIOD):
    signals = make_signals(data, n1=n1, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio, use_vwap=use_vwap, smooth_period=smooth_period)
    freq = data.index.to_series().diff().median()

    long_signal  = signals["long"].to_numpy()
    short_signal = signals["short"].to_numpy()
    # vbt wants float64 for size/close; cast just the columns it needs.
    close_np = data["Close"].to_numpy(dtype=np.float64)

    portfolio = vbt.Portfolio.from_signals(
        close=close_np,
        entries=long_signal,
        exits=short_signal,
        short_entries=short_signal,
        short_exits=long_signal,
        init_cash=CASH,
        fees=COMMISSION,
        size=100.0,
        size_type="value",
        freq=freq,
        sl_stop=signals["sl_pct"].to_numpy(dtype=np.float64),
        tp_stop=signals["tp_pct"].to_numpy(dtype=np.float64),
    )

    return portfolio, portfolio.stats(), signals

def run_params(data, n1, atr_period, sl_mult, rr_ratio, use_vwap=False, asset=None, timeframe=None, smooth_period=SMOOTH_PERIOD):
    try:
        portfolio, stats, _ = run_backtest(data, n1=n1, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio, use_vwap=use_vwap, smooth_period=smooth_period)
        return {
            "n1": n1, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "use_vwap": use_vwap,
            "commission": COMMISSION,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "return_pct": float(stats["Total Return [%]"]) if stats.get("Total Return [%]") is not None else None,
            "max_dd": float(stats["Max Drawdown [%]"]) if stats.get("Max Drawdown [%]") is not None else None,
            "sharpe": float(stats["Sharpe Ratio"]) if stats.get("Sharpe Ratio") is not None else None,
            "win_rate": float(stats["Win Rate [%]"]) if stats.get("Win Rate [%]") is not None else None,
            "profit_factor": float(stats["Profit Factor"]) if stats.get("Profit Factor") is not None else None,
            "expectancy": float(stats["Expectancy"]) if stats.get("Expectancy") is not None else None,
            "trades": int(stats["Total Trades"]) if stats.get("Total Trades") is not None else None,
        }
    except Exception as e:
        return {
            "n1": n1, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "use_vwap": use_vwap,
            "commission": COMMISSION,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "return_pct": None, "max_dd": None, "sharpe": None,
            "win_rate": None, "profit_factor": None,
            "expectancy": None, "trades": None,
        }


In [2]:
file_path = Path("~/market_data/XAUUSD/XAUUSD_M1.csv").expanduser()
data = load_data(file_path, verbose=True)

portfolio, stats, signals = run_backtest(data, rr_ratio=1)
print(stats)

trades = portfolio.trades.records_readable
print(trades)

free(data, signals)

[loader] cache hit: XAUUSD_M1__cache_v1.parquet
Start                                                 0
End                                             2088752
Period                               1450 days 12:33:00
Start Value                                     50000.0
End Value                                  38095.338224
Total Return [%]                             -23.809324
Benchmark Return [%]                         184.185378
Max Gross Exposure [%]                         0.262847
Total Fees Paid                            12213.951188
Max Drawdown [%]                              23.809324
Max Drawdown Duration                1450 days 12:14:00
Total Trades                                     407132
Total Closed Trades                              407131
Total Open Trades                                     1
Open Trade PnL                                -0.022643
Win Rate [%]                                  29.268712
Best Trade [%]                                 1.877104


In [ ]:
portfolio.plot()

## Optimization

In [3]:

from itertools import product
import time


db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    n1 INTEGER NOT NULL,
    atr_period INTEGER NOT NULL,
    sl_mult REAL NOT NULL,
    rr_ratio REAL NOT NULL,
    use_vwap INTEGER NOT NULL,
    commission REAL NOT NULL,
    smooth_period INTEGER NOT NULL,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    expectancy REAL,
    win_rate REAL,
    max_dd REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    num_trades INTEGER,
    return_pct REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

grid = list(product(
    ["M1", "M5", "M15"], # timeframes
    range(5, 15, 2), # fast ema,
    range(6, 16, 2), # smoothing
    [14, 18, 21], # atr period
    [1.0, 2.0, 3.0], # sl mult
    [0.25, 0.5, 1.0, 1.5, 2.0], # rr ratio
    [False, True], # use vwap
))

#* Only for test
# grid = list(product(
#     ["M1"], # timeframes
#     [9], # fast ema
#     [12], # smoothing
#     [14], # atr period
#     [1], # sl mult
#     [1], # rr ratio
#     [True, False], # use vwap
# ))

t0 = time.time()

print(f"[{time.strftime('%H:%M:%S')}] Testing {len(grid)} combinations -> {TABLE_NAME}")
asset = "XAUUSD"
# Load each timeframe once and reuse across all param combos (big memory win on M1).
data_by_tf: dict[tuple[str, bool], pd.DataFrame] = {}
for tf in sorted({g[0] for g in grid}):
    for uv in sorted({g[6] for g in grid if g[0] == tf}):
        fp = Path(f"~/market_data/{asset}/{asset}_{tf}.csv").expanduser()
        data_by_tf[(tf, uv)] = load_data(fp, verbose=True, use_vwap=uv)

ONE_LOOP_MEAN_TIME = 0.0
for i, (timeframe, n1, smoothing, ap, sm, rr, uv) in enumerate(grid):

    data = data_by_tf[(timeframe, uv)]

    t1 = time.time()
    result = run_params(data, n1, ap, sm, rr, use_vwap=uv, asset=asset, timeframe=timeframe, smooth_period=smoothing)
    elapsed = time.time() - t1
    cursor.execute(
        f"INSERT INTO {TABLE_NAME} (n1, atr_period, sl_mult, rr_ratio, use_vwap, commission, smooth_period, asset, timeframe, expectancy, win_rate, max_dd, sharpe_ratio, profit_factor, num_trades, return_pct) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (result["n1"], result["atr_period"], result["sl_mult"], result["rr_ratio"], uv, result["commission"], result["smooth_period"], result["asset"], result["timeframe"],
         result["expectancy"], result["win_rate"], result["max_dd"], result["sharpe"], result["profit_factor"], result["trades"], result["return_pct"]),
    )
    conn.commit()
    ONE_LOOP_MEAN_TIME = (elapsed + ONE_LOOP_MEAN_TIME * i) / (i + 1) if i else elapsed
    combinations_left = len(grid) - (i + 1)
    est_left_sec = ONE_LOOP_MEAN_TIME * combinations_left
    if (i + 1) % max(1, len(grid) // 100) == 0 or (i + 1) == len(grid):
        print(
            f"[{time.strftime('%H:%M:%S')}] {i+1:>5}/{len(grid)} tf={timeframe} n1={n1} ap={ap} sm={sm} rr={rr} vwap={uv} "
            f"| sharpe={result['sharpe']} ret={result['return_pct']} trades={result['trades']} ({elapsed:.1f}s) "
            f"| mean={ONE_LOOP_MEAN_TIME:.1f}s est_left={est_left_sec/60:.1f}min"
        )

    free()

free(data_by_tf)

sweep_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
sweep_df = sweep_df.dropna(subset=["sharpe_ratio"])
sweep_df = sweep_df.sort_values("sharpe_ratio", ascending=False).reset_index(drop=True)
print(f"Total grid: {len(grid)} | Non-null: {len(sweep_df)}")
sweep_df.head(20)



[17:29:44] Testing 6750 combinations -> f0_raw_smoothing
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[17:30:50]    67/6750 tf=M1 n1=5 ap=21 sm=1.0 rr=1.5 vwap=False | sharpe=-167.04257231749978 ret=-25.938075738024757 trades=446066 (0.9s) | mean=0.8s est_left=88.7min
[17:31:52]   134/6750 tf=M1 n1=5 ap=18 sm=2.0 rr=0.5 vwap=True | sharpe=-63.78087774327257 ret=-7.224453168007337 trades=126550 (0.7s) | mean=0.8s est_left=85.4min
[17:32:54]   201/6750 tf=M1 n1=5 ap=14 sm=3.0 rr=0.25 vwap=False | sharpe=-108.1831369708296 ret=-16.521964955776085 trades=290433 (0.8s) | mean=0.8s est_left=83.5min
[17:33:55]   268/6750 tf=M1 n1=5 ap=21 sm=3.0 rr=1.5 vwap=True | sharpe=-15.372828716930835 ret=-2.1899030587538992 trades=40197 (0.6s) | mean=0.8s est_lef

,id,n1,atr_period,sl_mult,rr_ratio,use_vwap,commission,smooth_period,asset,timeframe,expectancy,win_rate,max_dd,sharpe_ratio,profit_factor,num_trades,return_pct,run_timestamp
0,4860,5,21,3.0,2.0,1,0.00015,12,XAUUSD,M15,-0.012974,36.320327,0.140697,-0.864971,0.925068,4409,-0.113050,2026-08-18 11:19:44
1,4830,5,18,3.0,2.0,1,0.00015,12,XAUUSD,M15,-0.013961,36.277389,0.152960,-0.936000,0.919384,4428,-0.122277,2026-08-18 11:19:35
2,4770,5,21,3.0,2.0,1,0.00015,10,XAUUSD,M15,-0.014625,36.262991,0.160924,-0.963327,0.916975,4427,-0.128129,2026-08-18 11:19:18
3,4740,5,18,3.0,2.0,1,0.00015,10,XAUUSD,M15,-0.014794,36.283784,0.159460,-0.978800,0.915904,4441,-0.130038,2026-08-18 11:19:08
4,5310,7,21,3.0,2.0,1,0.00015,12,XAUUSD,M15,-0.014139,34.664948,0.157432,-1.010271,0.913604,4657,-0.130335,2026-08-18 11:21:59
5,4950,5,21,3.0,2.0,1,0.00015,14,XAUUSD,M15,-0.014999,35.458437,0.160534,-1.020330,0.911891,4440,-0.131828,2026-08-18 11:20:11
6,4858,5,21,3.0,1.5,1,0.00015,12,XAUUSD,M15,-0.014240,38.093193,0.161678,-1.043150,0.916050,4658,-0.132072,2026-08-18 11:19:44
7,5940,11,21,3.0,2.0,1,0.00015,6,XAUUSD,M15,-0.015195,35.228270,0.165942,-1.049845,0.910698,4557,-0.137123,2026-08-18 11:25:07
8,4768,5,21,3.0,1.5,1,0.00015,10,XAUUSD,M15,-0.014475,38.131798,0.165629,-1.050370,0.916394,4690,-0.135135,2026-08-18 11:19:17
9,4800,5,14,3.0,2.0,1,0.00015,12,XAUUSD,M15,-0.015842,36.042323,0.173078,-1.062194,0.908649,4443,-0.139410,2026-08-18 11:19:27


In [2]:
db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)

vwap_comparison = pd.read_sql_query(
    f"""SELECT use_vwap,
              COUNT(*) AS n,
              AVG(expectancy) AS mean_expectancy,
              AVG(win_rate) AS mean_win_rate,
              AVG(max_dd) AS mean_max_dd,
              AVG(sharpe_ratio) AS mean_sharpe,
              AVG(profit_factor) AS mean_profit_factor,
              AVG(num_trades) AS mean_num_trades,
              AVG(return_pct) AS mean_return_pct
       FROM {TABLE_NAME}
       WHERE sharpe_ratio IS NOT NULL
       GROUP BY use_vwap""",
    conn,
)
print(vwap_comparison)


   use_vwap     n  mean_expectancy  mean_win_rate  mean_max_dd  mean_sharpe  \
0         0  3375        -0.029071      48.223856     7.685517   -50.971418   
1         1  3375        -0.025947      42.283595     2.603094   -24.930563   

   mean_profit_factor  mean_num_trades  mean_return_pct  
0            0.520024    132786.192889        -7.683823  
1            0.617749     45804.814222        -2.596096  


In [ ]:
n1_comparison = pd.read_sql_query(
    """
    SELECT n1,
           COUNT(*) AS n,
           AVG(expectancy) AS mean_expectancy,
           AVG(win_rate) AS mean_win_rate,
           AVG(max_dd) AS mean_max_dd,
           AVG(sharpe_ratio) AS mean_sharpe,
           AVG(profit_factor) AS mean_profit_factor,
           AVG(num_trades) AS mean_num_trades,
           AVG(return_pct) AS mean_return_pct
    FROM """ + TABLE_NAME + """
    WHERE sharpe_ratio IS NOT NULL
    GROUP BY n1
    ORDER BY n1
    """,
    conn,
)
print(n1_comparison)

In [ ]:
db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)


n1_comparison_m5 = pd.read_sql_query(
    """
    SELECT n1,
           AVG(expectancy) AS mean_expectancy,
           AVG(win_rate) AS mean_win_rate,
           AVG(max_dd) AS mean_max_dd,
           AVG(sharpe_ratio) AS mean_sharpe,
           AVG(profit_factor) AS mean_profit_factor,
           AVG(num_trades) AS mean_num_trades,
           AVG(return_pct) AS mean_return_pct
    FROM """ + TABLE_NAME + """
    WHERE sharpe_ratio IS NOT NULL AND timeframe like 'M5'
    GROUP BY n1
    ORDER BY mean_win_rate, mean_expectancy
    """,
    conn,
)
print(n1_comparison_m5)